# Farmland Classification 

We used Open Street Map (OSM) for downloading farmland data. This shapefile is used in the BaseMap Jupyter Notebook to classify commercial-scale farming. You can read more about OSM farmaland data on https://wiki.openstreetmap.org/wiki/Tag:landuse=farmland. 

The original OSM shapefile specifies the crop and the system (irrigated vs rainfed) for each polygon. This Jupyter Notebook serves to create an additional column that creates crop classes that creates consistency with the IFPRI dataset. 

We used Zambia as an example. For instance, if a polygon had sugarcane and irrigation, than the class becomes SUGC_HI. Since the classification can vary greately depending on the country and engagement with local experts, this part needs to be manually edited by the user to match the context-specific needs. 

Here you will find the step-by-step guide on how to download farmland data from OSM on QGIS, and the classification code (to be revised depending on the classification assumptions).

The final output is a shapefile named farmlands_reclassified in the 00.InputData folder.

### Download farmland data from OSM

#### Step 1: Install QuickOSM Plugin
- Open QGIS.
- Select Plugins > Manage and Install Plugins
- Search for QuickOSM in the search bar.
- Click Install Plugin, then close the window.

Look for a small green magnifying glass icon  on the toolbar, or access it via Vector > QuickOSM > QuickOSM.

#### Step 2: Set Up the Query
- Open QuickOSM 
- Go to Quick query tab
- Set the parameters:
  Key: landuse
  Value: farmland
  
  Location Filter: 
  - Option 1 - In location: Type a town, region, or country name (e.g., Zambia).
  - Option 2 - Canvas Extent: Uses your current map view area in QGIS.

#### Output

The plug in returns a polygon, a vector, and a point layer.

**Export only the polygon** and rename it as *OSM_farmland*. **Make sure to choose the same projection system as your user settings in BaseMaps**

In [ ]:
import os
import re
import warnings
from typing import Dict, Tuple, Optional, List

import geopandas as gpd
import pandas as pd


from pathlib import Path
from Map.reclassify_farmlands import list_unique_crop, reclassify_farmlands




In [ ]:
ROOT_DIR = os.path.abspath(os.curdir)
OUT_NODATA = 255
Data_dir = os.path.join(ROOT_DIR, 'Data')
Input_data = os.path.join(Data_dir, '00.InputData')



### 1) List unique crops found in the shapefile

OSM farmland attribute tables have two key columns: one named crop and one irrigation. This function lists all the crops in the uploaded shapefile. Since each shapefile varies depending on the country, it is important to update the classification rule based on the crop list. This can be done by editing the user mapping dictionary in step 2. 

In [ ]:
# Print crop list
farmland_crop = list_unique_crop(str(Input_data))
print("Available crops in the `crop` column:")
for i, c in enumerate(farmland_crop, 1):
    print(f"{i:2d}. {c}")

## 2) Edit the classification rule in the USER_MAPPING. 
The OSM shapefile should have two key columns for farmland: crop and irrigation. The crop column specifies the type of crop, while irrigation could be NULL or "pivot" (which means that the crop is irrigated). In this cell the user specifies the farmalnd classification rule. Crop classification is needed to ensure consistency across the naming convention used in the final database produced in BaseMap, and the crops listed in OSM shapefile. Because of that, it is important to adopt a **consistent naming convention** with the one used by the **IFPRI** insitute (four upper case letter identifiers). You can check it in the 00.InputData/spam2020V2 folder. Then always keep HI and HR for High (which refers to comemrcial use) irrigated and rainfed systems.

The `USER_MAPPING` is a dictionary structured as "crop name in the OSM list": ("class name if irrigated", "class name if rainfed").
For instance, if in OSM crop is "sugarcane" and irrigation has "pivot", the workflow classifies this has SUGC_HI, where SUGC corresponds to the naming convention used for sugarcane and HI stands for "High-Irrigated"; if there is no "pivot", then the class becomes SUGC_HR (High-Rainfed). If the cell has no crops, then it falls under the class Other.

The words that mark a polygon as irrigated are set in `IRRIGATED_KEYWORDS` in the cell below. If your dataset uses other keywords, please change "pivot" or add more (e.g. "drip", "sprinkler"). 


In [ ]:
# 2) Classification rule: edit this cell to customize.
#
# Format:  "crop name as it appears in the OSM list": ("class if irrigated", "class if rainfed")
# - Matching is case-insensitive. If a crop string has no exact match, each part separated by
#   "," or ";" is tried, then each single word (e.g. "maize; beans" matches "maize").
# - The key None is used for polygons with an empty crop field.
# - Crop strings with no match get no class (they are listed after running step 3).

USER_MAPPING = {
    "sugarcane": ("SUGC_HI", "SUGC_HR"),
    "wheat": ("WHEA_HI", "WHEA_HI"),
    "soy": ("SOYB_HI", "SOYB_HR"),
    "soybean": ("SOYB_HI", "SOYB_HR"),
    "maize": ("MAIZ_HI", "MAIZ_HR"),
    "corn": ("MAIZ_HI", "MAIZ_HR"),
    "banana": ("BANA_HI", "BANA_HR"),
    "cassava": ("CASS_HI", "CASS_HR"),
    "coffee": ("COFF_HI", "COFF_HR"),
    "palm oil": ("SUNF_HR", "SUNF_HR"),
    "palm": ("SUNF_HR", "SUNF_HR"),
    "cereal": ("MILL_HI", "MILL_HR"),
    "vegetable": ("VEGE_HI", "VEGE_HR"),
    "vegetables": ("VEGE_HI", "VEGE_HR"),
    "vegetable;rape;vegetables;Papaya;mango;Guave": ("VEGE_HI", "VEGE_HR"),
    "wheat, soybean": ("WHEA_HI", "SOYB_HR"),
    "wheat,soyabean": ("WHEA_HI", "SOYB_HR"),
    "wheat, soyabean": ("WHEA_HI", "SOYB_HR"),
    "wheat; soybean": ("WHEA_HI", "SOYB_HR"),
    "wheat;soy bean": ("WHEA_HI", "SOYB_HR"),
    "wheat, soya": ("WHEA_HI", "SOYB_HR"),
    "wheat;soy": ("WHEA_HI", "SOYB_HR"),
    "wheat;soybean": ("WHEA_HI", "SOYB_HR"),
    None: ("Other_HI", "Other_HR"),
}

# A polygon is irrigated if its `irrigation` field contains any of these words (case-insensitive).
IRRIGATED_KEYWORDS = ["pivot"]

print("Irrigated if `irrigation` contains:", IRRIGATED_KEYWORDS)
print("Current USER_MAPPING:")
for k in sorted(USER_MAPPING.keys(), key=lambda x: ('' if x is None else str(x).lower())):
    print('-', k, '->', USER_MAPPING[k])


When ready, run the next cell to create `farmlands_reclassified.shp` in the
`Data/00.InputData` folder. This is the file that will be then used in the BaseMap workflow. The final output will have a column called "crop_class" with the classified cropland. 


In [ ]:
# 3) Run the reclassification and write output shapefile
gdf = reclassify_farmlands(
    str(Input_data),
    mapping=USER_MAPPING,
    irrigated_keywords=IRRIGATED_KEYWORDS,
    output_shp='farmlands_reclassified.shp',
)
print("Wrote reclassified shapefile:", os.path.join(Input_data, 'farmlands_reclassified.shp'))
print("Rows:", len(gdf))

# show a sample of the new column
if 'crop_class' in gdf.columns:
    print("Sample of the new `crop_class` column:")
    display(gdf[['crop', 'irrigation', 'crop_class']].head(10))

    # crop strings that matched no key in USER_MAPPING: add them to step 2 if needed
    unmatched = sorted(gdf.loc[gdf['crop_class'].isna() & gdf['crop'].notna(), 'crop'].astype(str).unique())
    if unmatched:
        print(f"{len(unmatched)} crop strings got no class (not in USER_MAPPING):")
        for c in unmatched:
            print(' -', c)
    else:
        print('All crop strings were matched by USER_MAPPING.')
else:
    print('Output column `crop_class` not found in result.')


### 4) List the new crop classes

Reads `farmlands_reclassified.shp` and lists every class in the `crop_class` column.


In [ ]:
# Print the new crop classes
farmland_classes = list_unique_crop(str(Input_data), input_shp='farmlands_reclassified.shp', field='crop_class')
print("Final classes in the `crop_class` column:")
for i, c in enumerate(farmland_classes, 1):
    print(f"{i:2d}. {c}")